
### Dataset: Laptop Price Dataset

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

sns.set(style='whitegrid')

In [ ]:

df = pd.read_csv('laptop_price.csv', encoding='latin1')
df.head()

---
## Task 1: Identify Data Quality Issues

In this task, we inspect the dataset to identify any data quality problems such as incorrect data types, inconsistent formats, or missing values.

In [ ]:

df.dtypes

In [ ]:

df.isna().sum()

In [ ]:

print('Ram samples:', df['Ram'].unique())
print('Weight samples:', df['Weight'].unique()[:10])

### Observations:
After inspecting the dataset, the following data quality issues were identified:

1. **`Ram` column** is stored as `object` (string) instead of numeric. It contains values like `'8GB'`, `'16GB'` — the unit `'GB'` must be removed before converting to a number.
2. **`Weight` column** is stored as `object` (string) instead of numeric. It contains values like `'1.37kg'` — the unit `'kg'` must be removed.
3. **No missing values** were detected in this dataset — all columns return 0 missing values.

These type issues must be fixed before any analysis or modeling can be performed.

In [ ]:
# Fix Ram column: remove 'GB' and convert to numeric
df['Ram'] = df['Ram'].str.replace('GB', '').astype(float)

# Fix Weight column: remove 'kg' and convert to numeric
df['Weight'] = df['Weight'].str.replace('kg', '').astype(float)

# Verify the fix
df.dtypes

`Ram` is now stored as `float64` and `Weight` is now stored as `float64`.
This allows us to perform numerical analysis, normalization, and outlier detection on these columns.

---
## Task 2: Apply One Missing Value Strategy and Explain Why

Since the original dataset has no missing values, we introduce artificial missing values for learning purposes.

In [ ]:
# Introduce artificial missing values in Price_euros
df2 = df.copy()
df2.loc[0:5, 'Price_euros'] = np.nan

print('Missing values after introduction:')
df2.isna().sum()

In [ ]:
df2.head(10)

### Chosen Strategy: Median Imputation

We choose **Median Imputation** for the `Price_euros` column.

**Reason:** The laptop price data contains outliers (very cheap and very expensive laptops). The mean is sensitive to these extreme values and may produce a misleading fill value. The **median** is more robust because it represents the middle value of the sorted data and is not affected by extreme prices. This makes it a safer and more accurate strategy for price-related data.

In [ ]:
# Apply Median Imputation
df_median = df2.copy()
df_median['Price_euros'].fillna(df_median['Price_euros'].median(), inplace=True)

print('Missing values after median imputation:')
print(df_median.isna().sum())

df_median.head(10)

The 6 missing values in `Price_euros` have been replaced with the median price.
The dataset size is preserved, and the fill value is not distorted by extreme laptop prices.

---
## Task 3: Detect and Handle Outliers Using IQR

We will detect and handle outliers in the `Price_euros` column using the Interquartile Range (IQR) method.

In [ ]:

plt.figure(figsize=(6, 4))
sns.boxplot(x=df['Price_euros'])
plt.title('Boxplot of Price_euros (Before Handling Outliers)')
plt.show()

In [ ]:

Q1 = df['Price_euros'].quantile(0.25)
Q3 = df['Price_euros'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f'Q1: {Q1}, Q3: {Q3}, IQR: {IQR}')
print(f'Lower bound: {lower:.2f}')
print(f'Upper bound: {upper:.2f}')

outliers = df[(df['Price_euros'] < lower) | (df['Price_euros'] > upper)]
print(f'\nNumber of outliers detected: {len(outliers)}')
outliers.head(10)

### Handling Strategy: Capping (Percentile Method)

Instead of removing the outliers completely, we use **Capping** to replace extreme values with the 5th and 95th percentile limits.

**Reason:** In a laptop price dataset, very high prices may represent premium/gaming laptops which are valid real-world transactions. Removing them entirely could lose important data. Capping keeps all records while reducing the effect of extreme values.

In [ ]:

lower_cap = df['Price_euros'].quantile(0.05)
upper_cap = df['Price_euros'].quantile(0.95)

df_capped = df.copy()
df_capped['Price_euros'] = df_capped['Price_euros'].clip(lower_cap, upper_cap)

print(f'Before capping - Max: {df["Price_euros"].max():.2f}, Min: {df["Price_euros"].min():.2f}')
print(f'After capping  - Max: {df_capped["Price_euros"].max():.2f}, Min: {df_capped["Price_euros"].min():.2f}')

In [ ]:

plt.figure(figsize=(6, 4))
sns.boxplot(x=df_capped['Price_euros'])
plt.title('Boxplot of Price_euros (After Capping)')
plt.show()

After capping, extreme laptop prices have been brought within a reasonable range.
The dataset size remains the same (1303 rows) and no records were lost.

---
## Task 4: Normalize Numerical Features Using Min-Max and Z-Score

We will normalize the `Price_euros` and `Ram` columns using both methods.

In [ ]:

df[['Price_euros', 'Ram']].head()

In [ ]:

scaler_minmax = MinMaxScaler()
df_minmax = df[['Price_euros', 'Ram']].copy()
df_minmax[['Price_euros', 'Ram']] = scaler_minmax.fit_transform(df_minmax)

print('After Min-Max Normalization (range 0 to 1):')
df_minmax.head()

After Min-Max normalization, all values are scaled between 0 and 1.
The cheapest laptop becomes 0 and the most expensive becomes 1.
This is useful for distance-based models like KNN and K-Means.

In [ ]:

scaler_zscore = StandardScaler()
df_zscore = df[['Price_euros', 'Ram']].copy()
df_zscore[['Price_euros', 'Ram']] = scaler_zscore.fit_transform(df_zscore)

print('After Z-Score Normalization (mean=0, std=1):')
df_zscore.head()

After Z-Score standardization, the data is centered around 0.
Values above average become positive, and values below average become negative.
This is useful for models like Linear Regression, SVM, and PCA.

---
## Task 5: Apply PCA Only if Features Show Correlation

Before applying PCA, we must check whether the numerical features are correlated.

In [ ]:

plt.figure(figsize=(6, 4))
sns.heatmap(df_zscore[['Price_euros', 'Ram']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap (Before PCA)')
plt.show()

### Correlation Analysis

The heatmap shows the correlation between `Price_euros` and `Ram`.

- A correlation value **close to 0.5 or above** suggests a moderate to strong relationship, which makes PCA useful.
- In this dataset, there is a **positive correlation** between Ram and Price — laptops with more RAM tend to be more expensive.

Since there is a meaningful correlation between the two features, applying PCA is justified.

In [ ]:

X = df_zscore[['Price_euros', 'Ram']]

pca = PCA(n_components=2)
principal_components = pca.fit_transform(X)

print('Explained Variance Ratio:', pca.explained_variance_ratio_)
print(f'PC1 captures {pca.explained_variance_ratio_[0]*100:.1f}% of the variance')
print(f'PC2 captures {pca.explained_variance_ratio_[1]*100:.1f}% of the variance')

In [ ]:

plt.figure(figsize=(6, 4))
plt.scatter(principal_components[:, 0], principal_components[:, 1], alpha=0.4)
plt.title('PCA Projection - Laptop Dataset')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()

### PCA Results

Each point represents one laptop.

- **PC1** (horizontal axis) captures the direction of maximum variance — it combines the Price and RAM information into one component.
- **PC2** (vertical axis) captures the remaining variance in the perpendicular direction.

If PC1 alone explains most of the variance (e.g., above 70%), then we can reduce from 2 features to 1 principal component while retaining most of the dataset's information.

This dimensionality reduction helps simplify models and reduce computational cost.